In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from src.utils import load_and_vectorize_data, translate_sentence
from src.transformer_model import Transformer

### Model Parameters

In [3]:
BATCH_SIZE = 32
D_MODEL = 128
NUM_HEADS = 4
D_FF = 512
NUM_LAYERS = 3
EPOCHS = 10
LEARNING_RATE = 0.0005

### Dataset And DataLoaders

In [4]:
enc_in, dec_in, dec_tar, src_vec, trg_vec, max_src, max_trg = load_and_vectorize_data()

Vectorization Ready. English Vocab: 3355, French Vocab: 7801
Max Source Length: 5, Max Target Length: 12


In [5]:
dataset = TensorDataset(enc_in, dec_in, dec_tar)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

### Model Initialization and Training

In [6]:
model = Transformer(
    src_vocab_size=src_vec.vocab_size,
    trg_vocab_size=trg_vec.vocab_size,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_layers=NUM_LAYERS
)

In [7]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [8]:
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    
    for batch_idx, (src, dec_i, dec_t) in enumerate(dataloader):
        optimizer.zero_grad()
        
        # outputs shape: (batch_size, seq_len, trg_vocab_size)
        outputs = model(src, dec_i)
        
        # Flatten target and prediction tensors for CrossEntropy
        loss = criterion(
            outputs.view(-1, trg_vec.vocab_size), 
            dec_t.view(-1)
        )
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] complete. Average Loss: {avg_loss:.4f}")

Epoch [1/10] complete. Average Loss: 4.2155
Epoch [2/10] complete. Average Loss: 2.7566
Epoch [3/10] complete. Average Loss: 2.0188
Epoch [4/10] complete. Average Loss: 1.5019
Epoch [5/10] complete. Average Loss: 1.1221
Epoch [6/10] complete. Average Loss: 0.8775
Epoch [7/10] complete. Average Loss: 0.7117
Epoch [8/10] complete. Average Loss: 0.6136
Epoch [9/10] complete. Average Loss: 0.5584
Epoch [10/10] complete. Average Loss: 0.5238


### Testing the Model

In [10]:
test_sentence = "I am honest"
translation = translate_sentence(
    sentence=test_sentence,
    model=model,
    src_vectorizer=src_vec,
    trg_vectorizer=trg_vec,
    max_trg_len=max_trg,
)

print(f"English: {test_sentence}")
print(f"French Translation: {translation}")

English: I am honest
French Translation: je suis honnête


### Observation and Coclusion
The results on the short sentences generally are very good as the first 20000 statements on which the model is trained are short sentences of three to max four words. As a result long sentences still have a disadvantage. The short sentences give a very good accuracy which is very good as compared to the previous projects using bare RNNs and LSTMs with only Bahdanau Attention